# Лекция 08. Цикл ML, ортогонализация, PCA и Ridge / ML cycle, orthogonalization, PCA and Ridge

**RU.** Четыре задания по 5–10 минут после соответствующих тем. Выполняйте сверху вниз на CPU в Google Colab. Заполните только ячейки «Ваш код». Не используйте готовые реализации вместо требуемого алгоритма. Примеры демонстрируют операции, но не содержат полного решения задания. Отдельный выполненный ноутбук ответов доступен только преподавателю.

**EN.** Four 5–10 minute exercises follow the corresponding topics. Run top to bottom on CPU in Google Colab. Fill only the student cells. Do not replace the requested algorithm with a library implementation. Examples demonstrate operations without giving the complete exercise solution. A separate executed answer notebook is private to the instructor.

## Данные / Data
[UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49), Alpaydin & Kaynak, CC BY 4.0; [sklearn load_digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html). 1797 реальных изображений цифр 8×8; наблюдение — изображение, 64 признака — интенсивности 0…16, цель — цифра 0…9. / 1,797 real 8×8 handwritten images; one observation is one image, 64 features are intensities 0…16, target is digit 0…9.

Нормировка на известную константу 16, stratified train/validation/test = 1078/359/360, seed 808. Среднее, PCA и веса обучаются только на train. Validation выбирает k и alpha. Test открываем после фиксации решения. Writer IDs недоступны; это не разбиение по авторам и не официальный benchmark. Эти же данные были в лекциях 06–07: новый seed не создаёт независимый новый тест. / Normalize by known constant 16; stratified split 1078/359/360, seed 808. Fit mean, PCA and weights on train only. Validation selects k and alpha. Open test only after freezing the choice. No writer IDs are available; this is neither writer-disjoint nor an official benchmark. Lectures 06–07 used the same data: a different seed does not create a fresh independent test.

Знаки PCA-компонент могут различаться; сравнивайте проекции, а не знаки векторов. / PCA component signs may differ; compare projections rather than vector signs.

In [ ]:
import numpy as np  # RU: операции над матрицами; EN: matrix operations.
import matplotlib.pyplot as plt  # RU: графики реальных изображений; EN: plots of real images.
from sklearn.datasets import load_digits  # RU: встроенные реальные цифры; EN: bundled real digit data.
from sklearn.model_selection import train_test_split  # RU: стратифицированное разбиение; EN: stratified splitting.
from sklearn.decomposition import PCA  # RU: независимый эталон проверки; EN: independent reference for checks.
from sklearn.linear_model import Ridge  # RU: эталон регуляризованной регрессии; EN: reference regularized regression.
digits = load_digits()  # RU: загружаем 1797 изображений; EN: load 1797 images.
X = digits.data.astype(float) / 16.0  # RU: используем известный диапазон яркости; EN: use the known intensity range.
y = digits.target  # RU: классы цифр 0–9; EN: digit classes 0–9.
ids = np.arange(len(y))  # RU: сохраняем номера наблюдений; EN: preserve observation IDs.
train, rest = train_test_split(ids, test_size=0.4, stratify=y, random_state=808)  # RU: выделяем обучающую часть; EN: separate training data.
valid, test = train_test_split(rest, test_size=0.5, stratify=y[rest], random_state=808)  # RU: делим остаток на выбор и оценку; EN: divide the remainder into selection and evaluation.
assert not (set(train) & set(valid) or set(train) & set(test) or set(valid) & set(test))  # RU: проверяем отсутствие пересечения; EN: verify disjoint IDs.
print(len(train), len(valid), len(test))  # RU: ожидаем 1078,359,360; EN: expect 1078,359,360.

## 1. Ортонормированный базис / Orthonormal basis · 5–10 min
**Материал / Explanation.** Скалярное произведение измеряет проекцию. Из каждого вектора последовательно вычитаем найденные направления и нормируем остаток. Это геометрическая ортогонализация; она не обещает независимости гиперпараметров ML. / A dot product measures a projection. Subtract existing directions and normalize the residual. This is geometric orthogonalization, not independence of ML hyperparameters.

**RU — задание.** Реализуйте `orthonormal_rows(V, tol=1e-10)` для вещественной матрицы `(m,d)`. Обрабатывайте строки в исходном порядке. Дважды пройдите по уже найденным единичным векторам, вычитая проекции. Остаток длиной ≤tol пропустите; остальные нормируйте. Верните `(rank,d)`, для полностью зависимого нулевого входа — `(0,d)`. Вход не изменять. Не используйте QR/SVD внутри решения. Проверьте на первых трёх обучающих изображениях, затем добавьте повтор первого изображения. Результат: `Q@Q.T≈I`, исходные строки восстанавливаются через `V@Q.T@Q`.

**EN — task.** Implement `orthonormal_rows(V, tol=1e-10)` for a real `(m,d)` matrix. Process rows in their original order. Make two projection-removal passes over existing unit vectors. Skip residuals with norm ≤tol and normalize the others. Return `(rank,d)`, including `(0,d)` for all-zero input. Do not mutate the input or call QR/SVD inside the solution. Test the first three training images, then append a duplicate of the first. Deliver a basis satisfying `Q@Q.T≈I` and reconstruction `V@Q.T@Q≈V`.

### Пример / Example

In [ ]:
v = X[train[0]].copy()  # RU: берём реальное изображение; EN: take a real image.
q = v / np.linalg.norm(v)  # RU: нормируем одно направление; EN: normalize one direction.
print(np.dot(X[train[1]], q))  # RU: измеряем координату второго изображения; EN: measure the second image coordinate.

### Ваш код / Your code

In [ ]:
def orthonormal_rows(V, tol=1e-10):
    raise NotImplementedError('Ваш код / Your code')

### Автопроверка / Check

In [ ]:
V = np.vstack([X[train[:3]], X[train[0]]])  # RU: добавляем реальную зависимую строку; EN: append a real dependent row.
original = V.copy()  # RU: сохраняем вход для проверки мутаций; EN: preserve input for mutation detection.
Q = orthonormal_rows(V)  # RU: запускаем студенческий алгоритм; EN: run the student algorithm.
assert Q.shape == (3, 64)  # RU: дубликат не увеличивает ранг; EN: a duplicate does not increase rank.
assert np.allclose(Q @ Q.T, np.eye(3), atol=1e-9)  # RU: проверяем ортонормальность; EN: check orthonormality.
assert np.allclose(V @ Q.T @ Q, V, atol=1e-9)  # RU: проверяем сохранение пространства; EN: check span preservation.
assert np.array_equal(V, original)  # RU: запрещаем изменение входа; EN: reject input mutation.
print('1: OK')  # RU: подтверждаем первый блок; EN: confirm block one.

## 2. PCA без утечки / PCA without leakage · 5–10 min
**Материал / Explanation.** PCA находит направления максимальной дисперсии центрированных данных. SVD даёт эти направления в строках Vᵀ. / PCA finds high-variance directions in centered data; SVD stores them in rows of Vᵀ.

**RU — задание.** Реализуйте `fit_pca(X_train,k)` и `project(X_new,mean,components)`. Вход конечный float `(n,d)`, `1≤k≤min(n,d)`, ненулевая общая дисперсия. Возврат: mean `(d,)`, components `(k,d)`, ratio `(k,)`. Используйте `np.linalg.svd(...,full_matrices=False)` и сумму квадратов ВСЕХ сингулярных значений в знаменателе ratio. В project только вычтите сохранённое mean и умножьте на components.T; запрещено обучаться на X_new. При k=16 визуализируйте первые две test-картинки и реконструкцию, только после настройки модели в задании 4; для текущей проверки используйте validation. Сравнивайте подпространства: знак компоненты не фиксирован.

**EN — task.** Implement `fit_pca(X_train,k)` and `project(X_new,mean,components)`. Inputs are finite float `(n,d)`, `1≤k≤min(n,d)`, with nonzero total variance. Return mean `(d,)`, components `(k,d)`, ratio `(k,)`. Use reduced SVD and the squared sum of ALL singular values as the ratio denominator. Project by subtracting stored mean and multiplying by components.T; never fit on X_new. Reconstruct validation images now; inspect the first two test images only after selection in exercise 4. Compare subspaces because component signs are arbitrary.

### Пример / Example

In [ ]:
preview_mean = X[train].mean(axis=0)  # RU: среднее обучающих изображений; EN: mean training image.
plt.imshow(preview_mean.reshape(8, 8), cmap='gray')  # RU: показываем среднюю цифру; EN: display the mean digit.
plt.axis('off')  # RU: убираем координатные оси; EN: hide coordinate axes.
plt.show()  # RU: выводим визуальную проверку; EN: render the visual check.

### Ваш код / Your code

In [ ]:
def fit_pca(X_train, k):
    raise NotImplementedError('Ваш код / Your code')
def project(X_new, mean, components):
    raise NotImplementedError('Ваш код / Your code')

### Автопроверка / Check

In [ ]:
mu, C, ratio = fit_pca(X[train], 16)  # RU: обучаем студенческое преобразование; EN: fit the student transform.
reference = PCA(n_components=16, svd_solver='full').fit(X[train])  # RU: независимое библиотечное сравнение; EN: independent library comparison.
assert np.allclose(mu, reference.mean_)  # RU: проверяем среднее train; EN: check the training mean.
assert np.allclose(C.T @ C, reference.components_.T @ reference.components_, atol=1e-8)  # RU: сравниваем подпространства без знака; EN: compare sign-invariant subspaces.
assert np.allclose(ratio, reference.explained_variance_ratio_)  # RU: проверяем знаменатель дисперсии; EN: verify total-variance denominator.
assert np.allclose(project(X[valid[:3]], mu, C), (X[valid[:3]] - mu) @ C.T)  # RU: проверяем перенос train-статистик; EN: verify reuse of training statistics.
print('2: OK')  # RU: подтверждаем второй блок; EN: confirm block two.

## 3. Ridge для классификации / Ridge classification · 5–10 min
**Материал / Explanation.** Минимизируем `||ZW+b−Y||²_F + alpha||W||²_F`, без деления loss на n; intercept не штрафуем. Y — one-hot, поэтому числа классов не задают расстояние между цифрами. / Minimize the summed squared error plus the weight penalty, with no division by n and no intercept penalty. One-hot targets avoid treating digit labels as distances.

**RU — задание.** Реализуйте `fit_ridge(Z,labels,alpha)` и `predict_ridge(Z,weights,intercept)`, alpha>0, labels — целые 0…9. Центрируйте Z и one-hot Y; решите `(Zc.T@Zc+alpha*I)W=Zc.T@Yc` через solve. Верните W `(k,10)` и b `(10,)`; прогноз — argmax по 10 выходам `(n,)`. При равенстве выбирается меньший индекс. Не используйте sklearn Ridge в решении: он нужен только как независимый эталон. Сравните прогнозы и коэффициенты с эталоном на 16 PCA-компонентах и alpha=1. Выходы не являются вероятностями.

**EN — task.** Implement `fit_ridge(Z,labels,alpha)` and `predict_ridge(Z,weights,intercept)`, with alpha>0 and integer labels 0…9. Center Z and one-hot Y; solve the stated system with `solve`. Return W `(k,10)` and b `(10,)`; predict `(n,)` via argmax across ten outputs, choosing the smaller index on ties. Do not use sklearn Ridge inside your solution; it is the independent oracle. Compare coefficients and predictions with the oracle on 16 PCA components at alpha=1. Outputs are scores, not probabilities.

### Пример / Example

In [ ]:
example_targets = np.eye(10)[y[train[:3]]]  # RU: кодируем три настоящих класса; EN: encode three real labels.
print(example_targets, example_targets.shape)  # RU: проверяем десять выходов; EN: verify ten outputs.

### Ваш код / Your code

In [ ]:
def fit_ridge(Z, labels, alpha):
    raise NotImplementedError('Ваш код / Your code')
def predict_ridge(Z, weights, intercept):
    raise NotImplementedError('Ваш код / Your code')

### Автопроверка / Check

In [ ]:
Z = project(X[train], mu, C)  # RU: используем предыдущий PCA; EN: use the previous PCA.
W, b = fit_ridge(Z, y[train], 1.0)  # RU: обучаем студентскую модель; EN: fit the student model.
ref = Ridge(alpha=1.0).fit(Z, np.eye(10)[y[train]])  # RU: эталон с тем же loss; EN: oracle with the same objective.
assert np.allclose(W, ref.coef_.T, atol=1e-8)  # RU: сверяем матрицу весов; EN: compare weight matrices.
assert np.allclose(b, ref.intercept_, atol=1e-8)  # RU: сверяем нешрафуемый intercept; EN: compare unpenalized intercepts.
assert np.array_equal(predict_ridge(Z, W, b), ref.predict(Z).argmax(axis=1))  # RU: проверяем ось argmax; EN: verify argmax axis.
print('3: OK')  # RU: подтверждаем третий блок; EN: confirm block three.

## 4. Выбор эксперимента / Experiment selection · 5–10 min
**Материал / Explanation.** Validation выбирает, test оценивает зафиксированное решение. Повторный подбор по test превращает его в validation. / Validation selects; test evaluates a frozen choice. Repeated test-based tuning turns test into validation.

**RU — задание.** Реализуйте `select_config(records)`: непустой список словарей k,alpha,val_accuracy. Максимизируйте val_accuracy, при точном равенстве выбирайте меньше k, затем больше alpha; не меняйте список. Выполните 20 сочетаний k∈{8,16,32,64}, alpha∈{.01,.1,1,10,100}; для каждого обучайте PCA и Ridge на train. Сохраните все строки в records. Зафиксируйте best, переобучите только на прежнем train и ровно один раз вычислите test_prediction. Напечатайте выбранные параметры и test accuracy. Не выбирайте по test; не меняйте сетку после его просмотра. Письменно объясните, почему k=64 допустимо и почему сравнение с CNN из другой лекции не является честным benchmark.

**EN — task.** Implement `select_config(records)` for a nonempty list with k,alpha,val_accuracy. Maximize validation accuracy; exact ties choose smaller k, then larger alpha. Do not mutate the list. Evaluate the 20 declared combinations, fitting PCA and Ridge on train for each. Store all records, freeze best, refit on the original train only, and compute test_prediction once. Print parameters and test accuracy. Do not select on test or revise the grid after viewing it. Explain why k=64 is valid and why comparison with a CNN from a different lecture is not a controlled benchmark.

### Пример / Example

In [ ]:
print('Candidate count:', 4 * 5)  # RU: фиксируем бюджет до эксперимента; EN: declare the budget before experimentation.
print('Test is reserved:', len(test))  # RU: показываем размер без просмотра меток; EN: show size without inspecting labels.

### Ваш код / Your code

In [ ]:
def select_config(records):
    raise NotImplementedError('Ваш код / Your code')
# RU: добавьте цикл эксперимента; EN: add the experiment loop.

### Автопроверка / Check

In [ ]:
assert len(records) == 20  # RU: проверяем полный заявленный бюджет; EN: verify the full declared budget.
assert best == sorted(records, key=lambda r: (-r['val_accuracy'], r['k'], -r['alpha']))[0]  # RU: проверяем правило выбора; EN: verify selection rule.
assert best['k'] == 64 and best['alpha'] == 1.0  # RU: сверяем воспроизводимый учебный запуск; EN: check the reproducible teaching run.
assert test_prediction.shape == (len(test),)  # RU: один прогноз на изображение; EN: one prediction per image.
assert np.mean(test_prediction == y[test]) > 0.9  # RU: отсеиваем неработающий полный pipeline; EN: reject a broken end-to-end pipeline.
print('4: OK')  # RU: подтверждаем четвёртый блок; EN: confirm block four.

## Обсуждение / Discussion
RU: k=64 сохраняет полную систему координат; PCA здесь центрирует и поворачивает данные. Изотропная L2-регуляризация при полном ортогональном повороте эквивалентна исходным координатам. Результат этого запуска: validation 337/359, test 338/360; это описание фиксированного эксперимента, а не цель для подгонки. Меньшие k теряют информацию. Сравнение с CNN требует одинаковых данных, split, бюджета настройки и метрики.

EN: k=64 retains the full coordinate system: PCA centers and rotates data. Isotropic L2 regularization is invariant under a full orthogonal rotation. This run yields validation 337/359 and test 338/360; these describe a frozen experiment rather than tuning targets. Smaller k discard information. Comparing with a CNN requires identical data, split, tuning budget and metric.